In [1]:
import json
import itertools
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit.library import QAOAAnsatz, StatePreparation
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import LinearEqualityToPenalty

In [2]:
#1.Upload Data

In [3]:

with open("../data/beginner/parametros_8_activos.json", "r") as f:
    datos = json.load(f)

n = datos["n_activos"]
tickers = datos["tickers"]
mu = np.array(datos["mu"])
sigma = np.array(datos["sigma"])

# Parámetros iniciales
K = 4        # Cuántos activos elegir
q = 0.5      # Aversión al riesgo
lam = 10.0   # Penalización inicial
lambdas_to_test = [1.0, 5.0, 10.0, 20.0, 50.0]
p_depths = [1, 2, 3, 4]

print(f"Optimizando portafolio para {n} activos: {tickers}")

Optimizando portafolio para 8 activos: ['AAPL', 'MSFT', 'NVDA', 'IBE.MC', 'BBVA.MC', 'CAF.MC', 'GOOGL', 'CIE.MC']


In [4]:
#2

In [5]:
def build_portfolio_qp(penalty_lam):
    qp = QuadraticProgram()
    for ticker in tickers:
        qp.binary_var(name=ticker)

    qp.maximize(linear=mu, quadratic=-q * sigma)
    qp.linear_constraint(linear={t: 1 for t in tickers}, sense="==", rhs=K, name="budget")

    qubo = LinearEqualityToPenalty(penalty=penalty_lam).convert(qp)
    return qp, qubo

In [6]:
# ==========================================
# 2. CONSTRUCCIÓN DE QUBO Y SOLUCIÓN EXACTA
# ==========================================
qp_ref, _ = build_portfolio_qp(penalty_lam=1.0)

best_exact, best_combo = -np.inf, None
for combo in itertools.combinations(range(n), K):
    x = [1 if i in combo else 0 for i in range(n)]
    val = qp_ref.objective.evaluate(x)
    if val > best_exact:
        best_exact, best_combo = val, x

optimal_value_exact = best_exact
print(f"Valor óptimo teórico: {optimal_value_exact:.4f}")
print(f"Cartera óptima: {[tickers[i] for i in range(n) if best_combo[i]]}")



Valor óptimo teórico: 2.2348
Cartera óptima: ['IBE.MC', 'BBVA.MC', 'CAF.MC', 'GOOGL']


In [7]:
def run_qaoa(H, qp, K, p, restarts=1, shots=4096, maxiter=100, seed=0,
             init_state=None, mixer=None, x0=None):
    # CAMBIO: normalizar el Hamiltoniano
    H = H * (1.0 / np.max(np.abs(H.coeffs)))

    kw = {}
    if init_state is not None: kw["initial_state"] = init_state
    if mixer is not None:      kw["mixer_operator"] = mixer

    ansatz_no_meas = QAOAAnsatz(cost_operator=H, reps=p, **kw)
    ansatz = ansatz_no_meas.copy()      # CAMBIO: copy en vez de construir dos veces
    ansatz.measure_all()

    estimator, sampler = StatevectorEstimator(), StatevectorSampler()

    def cost_function(params):
        pub = (ansatz_no_meas, H, params)
        return float(estimator.run([pub]).result()[0].data.evs)

    # CAMBIO: varios reinicios con semilla fija (+ warm start opcional en el primero)
    rng = np.random.default_rng(seed)
    best = None
    for r in range(restarts):
        if x0 is not None and r == 0:
            start = x0
        else:
            start = rng.uniform(0, np.pi, ansatz_no_meas.num_parameters)
        res = minimize(cost_function, start, method="COBYLA", options={"maxiter": maxiter})
        if best is None or res.fun < best.fun:
            best = res

    counts = sampler.run([(ansatz, best.x)], shots=shots).result()[0].data.meas.get_counts()

    # CAMBIO: evaluar TODOS los bitstrings, solo los factibles, con el objetivo ORIGINAL
    feasible_shots, best_x, best_val = 0, None, -np.inf
    for bitstring, c in counts.items():
        x = [int(b) for b in bitstring.replace(" ", "")[::-1]]
        if sum(x) == K:
            feasible_shots += c
            val = qp.objective.evaluate(x)
            if val > best_val:
                best_val, best_x = val, x

    return dict(x=best_x, value=best_val, p_feasible=feasible_shots / shots,
                params=best.x, energy=best.fun)

In [8]:
lam = 10.0
p_depth = 2

qp, qubo = build_portfolio_qp(lam)
H, offset = qubo.to_ising()

out = run_qaoa(H, qp, K, p_depth)

print(f"Resultado QAOA (p={p_depth}, lambda={lam}):")
print(f"Vector binario: {out['x']}")
if out["x"] is not None:
    print(f"Activos seleccionados: {[tickers[i] for i in range(n) if out['x'][i]]}")
    print(f"Valor obtenido: {out['value']:.4f}")
    print(f"Approximation Ratio: {out['value'] / optimal_value_exact:.4f}")
print(f"Probabilidad de muestras factibles: {out['p_feasible']:.2%}") #No mide la cartera, solo solo si respeta las resticciones

/home/oier/qiskit_fallfest/.venv/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/home/oier/qiskit_fallfest/.venv/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/home/oier/qiskit_fallfest/.venv/lib/python3.12/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Resultado QAOA (p=2, lambda=10.0):
Vector binario: [0, 0, 0, 1, 1, 1, 1, 0]
Activos seleccionados: ['IBE.MC', 'BBVA.MC', 'CAF.MC', 'GOOGL']
Valor obtenido: 2.2348
Approximation Ratio: 1.0000
Probabilidad de muestras factibles: 79.37%


In [ ]:
results_sweep = []

for lam_val in lambdas_to_test:
    qp, qubo = build_portfolio_qp(lam_val)
    H, _ = qubo.to_ising()
    prev = None

    for p in p_depths:
        # CAMBIO: warm start desde la profundidad anterior
        x0 = None
        if prev is not None:
            betas, gammas = prev[:len(prev)//2], prev[len(prev)//2:]
            x0 = np.concatenate([betas, [betas[-1]], gammas, [gammas[-1]]])

        out = run_qaoa(H, qp, K, p, x0=x0)
        prev = out["params"]

        ratio = out["value"] / optimal_value_exact if out["x"] is not None else 0.0
        results_sweep.append({"lambda": lam_val, "p": p,
                              "approx_ratio": ratio, "p_feasible": out["p_feasible"]})
        print(f"lambda={lam_val:5} p={p}  ratio={ratio:.4f}  feasible={out['p_feasible']:.1%}")

lambda=  1.0 p=1  ratio=1.0000  feasible=27.1%
lambda=  1.0 p=2  ratio=1.0000  feasible=36.8%
lambda=  1.0 p=3  ratio=1.0000  feasible=45.4%
lambda=  1.0 p=4  ratio=1.0000  feasible=48.0%
lambda=  5.0 p=1  ratio=1.0000  feasible=63.4%
lambda=  5.0 p=2  ratio=1.0000  feasible=78.0%
lambda=  5.0 p=3  ratio=1.0000  feasible=96.9%
lambda=  5.0 p=4  ratio=1.0000  feasible=99.6%
lambda= 10.0 p=1  ratio=1.0000  feasible=62.3%
lambda= 10.0 p=2  ratio=1.0000  feasible=85.4%


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for l_val in lambdas_to_test:
    subset = [r for r in results_sweep if r["lambda"] == l_val]
    ps = [r["p"] for r in subset]
    axes[0].plot(ps, [r["approx_ratio"] for r in subset], marker="o", label=f"lambda={l_val}")
    axes[1].plot(ps, [r["p_feasible"] for r in subset], marker="o", label=f"lambda={l_val}")

axes[0].set_title("Ratio de aproximación")
axes[1].set_title("Probabilidad de muestras factibles")
for ax in axes:
    ax.set_xlabel("Profundidad del circuito (p)")
    ax.set_xticks(p_depths)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend()
axes[0].set_ylabel("Ratio")
axes[1].set_ylabel("Fracción factible")
plt.tight_layout()
plt.show()

In [ ]:
#Final
